# 🧪 ToM-SLM-QLoRA — Tier 3 Follow-up Experiments

논문피드백 (`홍지형_논문피드백_학생전달용.txt`) 의 다음 세 항목을 다룹니다.

- **[필수 수정 D] 다중 시드 재학습** — seed 43, 44로 추가 학습 + 5개 벤치마크 재평가 → Δaccuracy 평균±표준편차
- **[중요 수정 F] few-shot/CoT 베이스라인** — 학습 없이 base 모델에 few-shot+CoT 예시만 추가
- **[필수 수정 C, C3] 포맷통제 비교** — base 모델에 파인튜닝과 동일한 출력 포맷(reasoning 없이 `[[X]]`)만 few-shot으로 제공, 포맷학습 효과 통제

**전제**: 기존 `tombench_full_pipeline.ipynb` 를 최소 한 번 실행해서 `results/`, `splits/` 가 Drive에 이미 있어야 seed=42 기준값과 비교할 수 있습니다.
이 노트북은 **독립 실행 가능**하도록 데이터 로드·분할을 처음부터 다시 수행합니다 (동일 seed=42, 동일 코드 → 동일 test set 보장).

**GPU 메모리 안내**: 시드마다 매번 모델을 새로 로드합니다 (LoRA 어댑터 재사용 불가 — 서로 다른 초기화가 핵심이므로). Colab에서 T4/L4 기준 시드당 학습 15~20분 + 평가 몇 분 정도 예상됩니다.


## ⚙️ [1/7] 환경 설정 (기존 파이프라인과 동일)

In [ ]:
import sys, os, subprocess, time
import torch
if not torch.cuda.is_available():
    print('\n❌ GPU 미설정. 런타임 → 런타임 유형 변경 → GPU'); raise SystemExit
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'✅ GPU: {gpu_name} ({vram_gb:.1f}GB VRAM)')

print('\n📦 라이브러리 설치 (2~3분)...')
t0 = time.time()
!pip install -q unsloth
!pip install -q "datasets<4.0.0" scikit-learn pandas tqdm
print(f'   ⏱ {time.time()-t0:.0f}초')

from google.colab import drive
drive.mount('/content/drive')
DRIVE_FOLDER = 'ToMBench_연구결과'   # 기존 파이프라인과 반드시 동일해야 seed=42 결과와 비교 가능
DRIVE_DIR = f'/content/drive/MyDrive/{DRIVE_FOLDER}'
TIER3_DIR = f'{DRIVE_DIR}/tier3'
os.makedirs(TIER3_DIR, exist_ok=True)
print(f'✅ Drive: {DRIVE_DIR}  |  Tier3 출력: {TIER3_DIR}')


## 📥 [2/7] ToMBench 로드 + 분할 (기존 파이프라인 [2/14]와 **완전히 동일한 코드**)

⚠️ 이 셀은 손대지 마세요. seed=42, TEST_RATIO=0.30, VAL_RATIO_OF_TRAIN=0.20 이 기존 논문 결과와 정확히 같은 test set을 보장하는 유일한 방법입니다.

In [ ]:
import json, glob, math, re, random
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

TOMBENCH_DATA_PATHS = [
    '/content/ToMBench/data', '/content/data',
    '/content/drive/MyDrive/ToMBench-main/data',
    '/content/drive/MyDrive/ToMBench/data',
]
OPENTOM_DATA_PATHS = [
    '/content/OpenToM/data',
    '/content/drive/MyDrive/OpenToM-main/data',
    '/content/drive/MyDrive/OpenToM/data',
]
TEST_RATIO = 0.30
VAL_RATIO_OF_TRAIN = 0.20
EVAL_MAX_PER_DATASET = 1000

TOMBENCH_DIR = None
for p in TOMBENCH_DATA_PATHS:
    if os.path.isdir(p) and glob.glob(os.path.join(p, '*.jsonl')):
        TOMBENCH_DIR = p; break
if TOMBENCH_DIR is None:
    subprocess.run(['git','clone','--depth','1','https://github.com/zhchen18/ToMBench.git','/content/ToMBench'], check=False)
    TOMBENCH_DIR = '/content/ToMBench/data'
assert os.path.isdir(TOMBENCH_DIR)
print(f'✅ ToMBench: {TOMBENCH_DIR}')

def is_valid(x):
    if x is None: return False
    if isinstance(x, float) and math.isnan(x): return False
    s = str(x).strip()
    return len(s) > 0 and s.lower() != 'nan'

def normalize_ability(ab):
    return ab.replace('Non-literal', 'Non-Literal')

def build_user_prompt(story, question, opts):
    lines = [f'{l}. {t}' for l, t in opts]
    return (f'[Story]\n{story}\n\n[Question]\n{question}\n\n[Candidate Answers]\n' + '\n'.join(lines))

records = []
for fp in sorted(glob.glob(os.path.join(TOMBENCH_DIR, '*.jsonl'))):
    task = os.path.splitext(os.path.basename(fp))[0]
    with open(fp, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            row = json.loads(line)
            ability = normalize_ability(row.get('能力\nABILITY', 'Unknown'))
            category = ability.split(':')[0].strip()
            ans = row.get('答案\nANSWER') or row.get('ANSWER')
            if not is_valid(row.get('STORY')) or not is_valid(row.get('QUESTION')) or not is_valid(ans): continue
            opts = []
            for letter in ['A','B','C','D']:
                v = row.get(f'OPTION-{letter}')
                if is_valid(v): opts.append((letter, str(v).strip()))
            records.append({
                'source': 'ToMBench', 'task': task, 'category': category, 'ability': ability,
                'user_prompt': build_user_prompt(row['STORY'], row['QUESTION'], opts),
                'answer': str(ans).strip().upper()[0],
            })
df = pd.DataFrame(records)
print(f'✅ ToMBench 총 {len(df)}개')

SMALL_THRESHOLD = 30
ab_counts = df['ability'].value_counts()
small_ab = ab_counts[ab_counts < SMALL_THRESHOLD].index.tolist()
large_ab = ab_counts[ab_counts >= SMALL_THRESHOLD].index.tolist()
large_df = df[df['ability'].isin(large_ab)].reset_index(drop=True)
train_val_l, test_l = train_test_split(large_df, test_size=TEST_RATIO, random_state=42, stratify=large_df['ability'])
train_l, val_l = train_test_split(train_val_l, test_size=VAL_RATIO_OF_TRAIN, random_state=42, stratify=train_val_l['ability'])
parts = {'train': [], 'val': [], 'test': []}
for ab in small_ab:
    ab_df = df[df['ability'] == ab].reset_index(drop=True)
    n = len(ab_df)
    if n < 5:
        parts['train'].append(ab_df); continue
    seed_ab = 42 + (hash(ab) % 100)
    tr, vt = train_test_split(ab_df, test_size=0.4, random_state=seed_ab)
    va, te = train_test_split(vt, test_size=0.5, random_state=seed_ab)
    parts['train'].append(tr); parts['val'].append(va); parts['test'].append(te)
train_s = pd.concat(parts['train']) if parts['train'] else pd.DataFrame()
val_s = pd.concat(parts['val']) if parts['val'] else pd.DataFrame()
test_s = pd.concat(parts['test']) if parts['test'] else pd.DataFrame()
train_df = pd.concat([train_l, train_s], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
val_df   = pd.concat([val_l, val_s],     ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
test_df  = pd.concat([test_l, test_s],   ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
print(f'✅ Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}  (논문 표와 반드시 1607/411/842 이어야 함)')
assert len(train_df)==1607 and len(val_df)==411 and len(test_df)==842, '⚠️ split 크기가 논문과 다릅니다 — 데이터/코드 버전을 확인하세요'


## 🤖 [3/7] 모델 로드 헬퍼 + 평가 헬퍼 (기존과 동일 SYSTEM_PROMPT·추출 규칙)

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
MAX_SEQ_LENGTH = 2048

SYSTEM_PROMPT = (
    'Below is a multiple-choice question with a story and several answer options. '
    'Based on the content of the story and the given question, please infer the most likely answer '
    'and output the answer index in the format [[X]] where X is one of A, B, C, D, E.'
)

ANS_RE = re.compile(r'\[\[\s*([A-E])\s*\]\]', re.IGNORECASE)
FB_RE = re.compile(r'\b([A-E])\b')
def extract_letter(text):
    m = ANS_RE.search(text)
    if m: return m.group(1).upper()
    m = FB_RE.search(text)
    if m: return m.group(1).upper()
    return 'A'

def to_text(tokenizer, user_prompt, few_shot_prefix=None):
    msgs = [{'role':'system','content':SYSTEM_PROMPT}]
    if few_shot_prefix:
        msgs += few_shot_prefix
    msgs.append({'role':'user','content':user_prompt})
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def evaluate(eval_records, model, tokenizer, desc='Eval', few_shot_prefix=None):
    from tqdm.auto import tqdm
    FastLanguageModel.for_inference(model)
    results = []
    for r in tqdm(eval_records, desc=desc):
        prompt = to_text(tokenizer, r['user_prompt'], few_shot_prefix=few_shot_prefix)
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        out = model.generate(**inputs, max_new_tokens=12, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
        gen = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        pred = extract_letter(gen)
        results.append({**r, 'pred': pred, 'raw': gen, 'correct': int(pred == r['answer'])})
    return pd.DataFrame(results)

def load_fresh_model():
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH, dtype=None, load_in_4bit=True)
    return model, tokenizer

print('✅ 헬퍼 준비 완료')


## 📦 [4/7] 외부 벤치마크 로더 (기존 파이프라인 [4/14]에서 그대로 복사)

⚠️ 이 셀도 기존과 동일해야 합니다. (내용 생략 없이 원본 그대로) — `load_opentom`, `load_tomi`, `load_socialiqa`, `load_hitom`, `cap_eval` 정의.

In [ ]:
def cap_eval(recs, n=EVAL_MAX_PER_DATASET):
    if n and len(recs) > n: return random.Random(42).sample(recs, n)
    return recs

# --- OpenToM ---
def load_opentom():
    OPENTOM_DIR = None
    for p in OPENTOM_DATA_PATHS:
        if os.path.isdir(p): OPENTOM_DIR = p; break
    if OPENTOM_DIR is None:
        subprocess.run(['git','clone','--depth','1','https://github.com/seacowx/OpenToM.git','/content/OpenToM'], check=False)
        if os.path.isdir('/content/OpenToM/data'): OPENTOM_DIR = '/content/OpenToM/data'
    if OPENTOM_DIR is None: print('⚠️ OpenToM 없음'); return []
    meta_path = os.path.join(OPENTOM_DIR, 'opentom_data', 'meta_data.json')
    recs = []
    if os.path.exists(meta_path):
        with open(meta_path) as f: meta = json.load(f)
        att_path = os.path.join(OPENTOM_DIR, 'opentom_data', 'attitude.json')
        if os.path.exists(att_path):
            with open(att_path) as f: att = json.load(f)
            opts_text = ['positive','negative','neutral']
            for sid, qs in att.items():
                narrative = meta.get(sid,{}).get('narrative','')
                if not narrative: continue
                for q in qs:
                    gold = q.get('answer','').strip().lower()
                    if gold not in opts_text: continue
                    shuffled = opts_text[:]
                    random.Random((hash(sid+q.get('question','')) & 0xffffffff)).shuffle(shuffled)
                    gold_letter = 'ABC'[shuffled.index(gold)]
                    recs.append({
                        'source':'OpenToM','task':'attitude','category':'Emotion/Attitude','ability':'Attitude',
                        'user_prompt':build_user_prompt(narrative, q['question'], list(zip('ABC',shuffled))),
                        'answer':gold_letter,
                    })
        cg_path = os.path.join(OPENTOM_DIR, 'opentom_data', 'location_cg_fo.json')
        if os.path.exists(cg_path):
            with open(cg_path) as f: cg = json.load(f)
            for sid, qs in cg.items():
                narrative = meta.get(sid,{}).get('narrative','')
                if not narrative: continue
                for q in qs:
                    gold = q.get('answer','').strip()
                    if gold not in ('Yes','No'): continue
                    opts = [('A','Yes'),('B','No')]
                    gold_letter = 'A' if gold == 'Yes' else 'B'
                    recs.append({
                        'source':'OpenToM','task':'location-cg','category':'Belief','ability':'Location False Beliefs',
                        'user_prompt': build_user_prompt(narrative, q['question'], opts),
                        'answer': gold_letter,
                    })
    return cap_eval(recs)

# --- ToMi ---
def load_tomi():
    TOMI_DIR = '/content/ToMi'
    if not os.path.isdir(TOMI_DIR):
        r = subprocess.run(['git','clone','--depth','1','https://github.com/facebookresearch/ToMi.git', TOMI_DIR], capture_output=True)
        if r.returncode != 0: print('⚠️ ToMi 클론 실패'); return []
    DATA_OUT = '/content/tomi_data'
    os.makedirs(DATA_OUT, exist_ok=True)
    if not os.path.exists(os.path.join(DATA_OUT,'test.txt')):
        r = subprocess.run([sys.executable,'main.py','-n','500','-o',DATA_OUT,'-s','42'], cwd=TOMI_DIR, capture_output=True)
        if r.returncode != 0: print('⚠️ ToMi 생성 실패'); return []
    samples = []; cur_facts, cur_qas = [], []; prev = 0
    with open(os.path.join(DATA_OUT,'test.txt')) as f:
        for line in f:
            line = line.rstrip('\n')
            if not line.strip(): continue
            m = re.match(r'^(\d+)\s(.*)$', line)
            if not m: continue
            num = int(m.group(1)); rest = m.group(2)
            if num <= prev and (cur_facts or cur_qas):
                narr = ' '.join(cur_facts)
                for q,a in cur_qas: samples.append({'narrative':narr,'question':q,'answer':a})
                cur_facts, cur_qas = [], []
            prev = num
            if '\t' in rest:
                parts = rest.split('\t')
                if len(parts) >= 2: cur_qas.append((parts[0].strip(), parts[1].strip()))
            else:
                cur_facts.append(rest.strip())
        if cur_facts or cur_qas:
            narr = ' '.join(cur_facts)
            for q,a in cur_qas: samples.append({'narrative':narr,'question':q,'answer':a})
    ans_pool = list({s['answer'] for s in samples})
    if len(ans_pool) < 4: return []
    recs = []
    for s in samples:
        gold = s['answer']
        distractors = [a for a in ans_pool if a != gold]
        ds = random.Random(hash(s['narrative'][:40]) & 0xffffffff).sample(distractors, 3)
        opts = [gold] + ds
        random.Random(hash(s['question']) & 0xffffffff).shuffle(opts)
        gold_letter = 'ABCD'[opts.index(gold)]
        recs.append({
            'source':'ToMi','task':'location-belief','category':'Belief','ability':'Location False Beliefs',
            'user_prompt': build_user_prompt(s['narrative'], s['question'], list(zip('ABCD', opts))),
            'answer': gold_letter,
        })
    return cap_eval(recs)

# --- SocialIQa ---
def load_socialiqa():
    from datasets import load_dataset
    ds = None
    for name in ['allenai/social_i_qa','social_i_qa']:
        try:
            ds = load_dataset(name, split='validation'); print(f'✅ SocialIQa from HF: {name}'); break
        except Exception:
            try:
                ds = load_dataset(name, split='validation', trust_remote_code=True); print(f'✅ SocialIQa from HF (legacy): {name}'); break
            except Exception as e2:
                print(f'  ↳ {name}: {type(e2).__name__}'); continue
    if ds is None:
        try:
            import urllib.request, zipfile, glob as _glob
            url = 'https://storage.googleapis.com/ai2-mosaic/public/socialiqa/socialiqa-train-dev.zip'
            zip_path = '/content/socialiqa.zip'
            urllib.request.urlretrieve(url, zip_path)
            with zipfile.ZipFile(zip_path) as z: z.extractall('/content/socialiqa_data')
            dev_jsonl = _glob.glob('/content/socialiqa_data/**/dev.jsonl', recursive=True)
            dev_lbl   = _glob.glob('/content/socialiqa_data/**/dev-labels.lst', recursive=True)
            if dev_jsonl and dev_lbl:
                with open(dev_jsonl[0]) as f: rows = [json.loads(l) for l in f]
                with open(dev_lbl[0]) as f: lbls = [l.strip() for l in f]
                ds = [dict(r, label=lbls[i]) for i,r in enumerate(rows)]
                print(f'✅ SocialIQa from AllenAI zip: {len(ds)}개')
        except Exception as e:
            print(f'  ↳ AllenAI zip 실패: {e}')
    if ds is None: print('⚠️ SocialIQa 로드 완전 실패 (스킵)'); return []
    recs = []
    for x in ds:
        ctx = x.get('context',''); q = x.get('question','')
        a,b,c = x.get('answerA',''), x.get('answerB',''), x.get('answerC','')
        lbl = str(x.get('label','')).strip()
        if lbl not in ('1','2','3'): continue
        gold_letter = {'1':'A','2':'B','3':'C'}[lbl]
        recs.append({
            'source':'SocialIQa','task':'social_cs','category':'Mixed','ability':'Mixed',
            'user_prompt': build_user_prompt(ctx, q, [('A',a),('B',b),('C',c)]),
            'answer': gold_letter,
        })
    return cap_eval(recs)

# --- HiToM ---
import re as _re_hitom
def _parse_hitom_choices(choices_str):
    if not isinstance(choices_str, str): return []
    pat = _re_hitom.compile(r'([A-Z])\.\s*([^,]+?)(?=,\s*[A-Z]\.|$)')
    return [(m.group(1).upper(), m.group(2).strip()) for m in pat.finditer(choices_str)]

def load_hitom():
    from datasets import load_dataset
    ds = None
    for name in ['Hi-ToM/Hi-ToM_Dataset', 'umwyf/Hi-ToM_Dataset']:
        try:
            ds = load_dataset(name, split='train'); print(f'✅ HiToM from {name}'); break
        except Exception as e:
            print(f'  ↳ {name}: {type(e).__name__}')
    if ds is None:
        try:
            HITOM_DIR = '/content/Hi-ToM_dataset'
            if not os.path.isdir(HITOM_DIR):
                subprocess.run(['git','clone','--depth','1','https://github.com/ying-hui-he/Hi-ToM_dataset.git', HITOM_DIR], capture_output=True, check=False)
            import glob as _glob
            json_files = _glob.glob(f'{HITOM_DIR}/**/*.json', recursive=True) + _glob.glob(f'{HITOM_DIR}/**/*.jsonl', recursive=True)
            if json_files:
                rows = []
                for fp in json_files:
                    with open(fp) as f:
                        try: data = json.load(f); rows.extend(data if isinstance(data, list) else [data])
                        except Exception: f.seek(0); rows.extend(json.loads(l) for l in f if l.strip())
                if rows: ds = rows; print(f'✅ HiToM from GitHub clone: {len(rows)}개')
        except Exception as e:
            print(f'  ↳ GitHub clone 실패: {e}')
    if ds is None: print('⚠️ HiToM 자동 로드 실패 (스킵)'); return []

    recs = []
    skipped = 0
    LETTERS = 'ABCDEFGHIJKLMNO'
    for x in ds:
        story = x.get('story') or x.get('context') or x.get('narrative') or ''
        q = x.get('question') or ''
        choices_raw = x.get('choices') or x.get('options') or []
        gold_raw = x.get('answer')
        if gold_raw is None: gold_raw = x.get('label')
        if not story or not q or not choices_raw or gold_raw is None: skipped += 1; continue
        if isinstance(choices_raw, str):
            opts = _parse_hitom_choices(choices_raw)
            if not opts:
                try:
                    parsed = json.loads(choices_raw); opts = list(zip(LETTERS, parsed))
                except Exception:
                    parts = [p.strip() for p in choices_raw.split('|')]; opts = list(zip(LETTERS, parts))
        elif isinstance(choices_raw, list):
            opts = list(zip(LETTERS, choices_raw))
        else:
            skipped += 1; continue
        if not opts: skipped += 1; continue
        gold_str = str(gold_raw).strip()
        gold_letter = None
        if len(gold_str) == 1 and gold_str.upper() in LETTERS:
            gold_letter = gold_str.upper()
        else:
            for letter, text in opts:
                if text.lower() == gold_str.lower(): gold_letter = letter; break
            if gold_letter is None:
                for letter, text in opts:
                    if gold_str.lower() in text.lower() or text.lower() in gold_str.lower(): gold_letter = letter; break
        if gold_letter is None: skipped += 1; continue
        if len(opts) > 5:
            gold_opt = next(o for o in opts if o[0] == gold_letter)
            others = [o for o in opts if o[0] != gold_letter]
            sampled = random.Random(hash(story+q) & 0xffffffff).sample(others, min(4, len(others)))
            new_pool = [gold_opt] + sampled
            random.Random(hash(q) & 0xffffffff).shuffle(new_pool)
            opts = [('ABCDE'[i], t) for i,(_, t) in enumerate(new_pool)]
            for i,(_, t) in enumerate(new_pool):
                if t == gold_opt[1]: gold_letter = 'ABCDE'[i]; break
        order = x.get('question_order', None)
        ability_label = f'High-Order False Beliefs (order={order})' if order is not None else 'High-Order False Beliefs'
        recs.append({
            'source':'HiToM','task':'high-order','category':'Belief','ability':ability_label,
            'user_prompt': build_user_prompt(story, q, opts),
            'answer': gold_letter,
        })
    if skipped: print(f'  ↳ HiToM: {skipped}개 스킵')
    return cap_eval(recs)

eval_sets = {'ToMBench': test_df.to_dict('records')}
eval_sets['OpenToM'] = load_opentom()
eval_sets['ToMi'] = load_tomi()
eval_sets['SocialIQa'] = load_socialiqa()
eval_sets['HiToM'] = load_hitom()
eval_sets = {k:v for k,v in eval_sets.items() if len(v) > 0}
print('📌 평가 예정 데이터셋:', {k: len(v) for k,v in eval_sets.items()})


## 🔁 [5/7] [필수 D] 다중 시드 재학습 (seed 43, 44 추가)

기존 파이프라인은 `seed=42` (LoRA 초기화 + Trainer 셔플) 로 한 번만 학습했습니다. 여기서는 데이터 분할(train/val/test)은 그대로 두고, **학습 시드만** 바꿔 2회 더 학습·평가합니다.

자원이 부담되면 `EXTRA_SEEDS = [43]` 로 1개만 돌려도 됩니다 (교수님 가이드: "2개라도 추가 권장").

In [ ]:
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments, EarlyStoppingCallback
from unsloth import is_bfloat16_supported

EXTRA_SEEDS = [43, 44]   # 자원 부담되면 [43] 로 줄이세요
NUM_EPOCHS = 4
EARLY_STOPPING_PATIENCE = 2

def to_sft_text(tokenizer, user_prompt, gold):
    msgs = [{'role':'system','content':SYSTEM_PROMPT}, {'role':'user','content':user_prompt},
            {'role':'assistant','content':f'[[{gold}]]'}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)

def train_one_seed(seed):
    print('='*60); print(f'🔥 시드 {seed} 학습 시작'); print('='*60)
    model, tokenizer = load_fresh_model()
    model = FastLanguageModel.get_peft_model(
        model, r=16,
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        lora_alpha=16, lora_dropout=0.0, bias='none',
        use_gradient_checkpointing='unsloth', random_state=seed,
    )
    _gpu = torch.cuda.get_device_name(0)
    _vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    if 'A100' in _gpu or _vram >= 35: TRAIN_BS, GRAD_ACCUM, EVAL_BS = 8, 1, 16
    elif 'L4' in _gpu or _vram >= 22: TRAIN_BS, GRAD_ACCUM, EVAL_BS = 4, 2, 8
    else: TRAIN_BS, GRAD_ACCUM, EVAL_BS = 2, 4, 4

    train_texts = [to_sft_text(tokenizer, r['user_prompt'], r['answer']) for r in train_df.to_dict('records')]
    val_texts   = [to_sft_text(tokenizer, r['user_prompt'], r['answer']) for r in val_df.to_dict('records')]
    train_ds_text = Dataset.from_dict({'text': train_texts})
    val_ds_text   = Dataset.from_dict({'text': val_texts})

    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer, train_dataset=train_ds_text, eval_dataset=val_ds_text,
        dataset_text_field='text', max_seq_length=MAX_SEQ_LENGTH, dataset_num_proc=2, packing=False,
        args=TrainingArguments(
            per_device_train_batch_size=TRAIN_BS, per_device_eval_batch_size=EVAL_BS,
            gradient_accumulation_steps=GRAD_ACCUM, warmup_ratio=0.03, num_train_epochs=NUM_EPOCHS,
            learning_rate=2e-4, fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
            logging_steps=20, eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
            metric_for_best_model='eval_loss', greater_is_better=False, save_total_limit=2,
            optim='adamw_8bit', weight_decay=0.01, lr_scheduler_type='cosine', seed=seed,
            output_dir=f'outputs_seed{seed}', report_to='none',
        ),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    )
    t0 = time.time()
    trainer.train()
    train_time = time.time()-t0
    print(f'✅ 시드 {seed} 학습 완료 ({train_time/60:.1f}분)')

    seed_results = {}
    for name, recs in eval_sets.items():
        res_df = evaluate(recs, model, tokenizer, desc=f'seed{seed}-{name}')
        res_df.to_csv(f'{TIER3_DIR}/seed{seed}_finetuned_{name}.csv', index=False)
        acc = res_df['correct'].mean()*100
        seed_results[name] = acc
        print(f'  {name}: acc={acc:.2f}%')
    seed_results['_train_time_sec'] = train_time

    del model, tokenizer, trainer
    torch.cuda.empty_cache()
    return seed_results

multi_seed_results = {}
for s in EXTRA_SEEDS:
    multi_seed_results[s] = train_one_seed(s)

print('\n📊 다중 시드 결과:')
print(json.dumps(multi_seed_results, indent=2, ensure_ascii=False))
with open(f'{TIER3_DIR}/multi_seed_results.json','w') as f:
    json.dump(multi_seed_results, f, indent=2, ensure_ascii=False)


### 📊 [5-1/7] 시드 통합 — Δaccuracy 평균 ± 표준편차

`SEED42_BASE_ACC` 는 기존 `results/SUMMARY_main_table.csv` (Baseline_Acc) 값을 그대로 넣었습니다 — base 모델은 시드에 영향받지 않으므로 재평가할 필요 없습니다.

In [ ]:
# 기존 파이프라인 결과 (results/SUMMARY_main_table.csv 값, 논문 Table 4와 동일)
BASE_ACC = {'ToMBench': 61.52, 'OpenToM': 63.10, 'ToMi': 75.30, 'SocialIQa': 67.40, 'HiToM': 63.00}
SEED42_TUNED_ACC = {'ToMBench': 76.13, 'OpenToM': 63.30, 'ToMi': 80.20, 'SocialIQa': 62.90, 'HiToM': 58.00}

rows = []
for ds in eval_sets.keys():
    deltas = [SEED42_TUNED_ACC[ds] - BASE_ACC[ds]]
    for s in EXTRA_SEEDS:
        if ds in multi_seed_results.get(s, {}):
            deltas.append(multi_seed_results[s][ds] - BASE_ACC[ds])
    rows.append({
        'dataset': ds, 'n_seeds': len(deltas),
        'delta_mean': float(np.mean(deltas)), 'delta_std': float(np.std(deltas, ddof=1)) if len(deltas)>1 else 0.0,
        'deltas_per_seed': deltas,
    })
multi_seed_summary = pd.DataFrame(rows)
print(multi_seed_summary.to_string(index=False))
multi_seed_summary.to_csv(f'{TIER3_DIR}/multi_seed_summary.csv', index=False)
print('\n💾 저장: multi_seed_summary.csv — 이 표를 논문 Limitations/D 대응 문단에 반영하세요.')


## 🧬 [Extra — A 최선책] Story-level Group Split 재학습 (데이터 누수 직접 검증)

기존 item-level split은 story 단위 경계를 무시합니다 (Limitations에 추가한 분석: story 55.1%가 문항 2개 이상 공유, 재현 시 test의 약 60%가 train과 story 공유).

여기서는 **같은 story를 가진 문항을 전부 같은 split에 배정**하는 `GroupShuffleSplit` 으로 train/val/test를 다시 만들고, 그 위에서 재학습·재평가합니다. 이 결과와 기존 +14.61%p를 비교하면 "누수 없이도 향상이 유지되는가"에 직접 답할 수 있습니다.

⚠️ ability 층화는 이 분할에서는 적용하지 않습니다 (group 제약 + 층화를 동시에 만족시키려면 별도 알고리즘이 필요해서, 우선 순수 group split으로 누수 여부만 검증). ability 분포가 원래 분할과 다소 달라질 수 있습니다.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# story 텍스트를 다시 확보 (df 재구성 시 user_prompt에서 [Story]...[Question] 사이 추출)
def extract_story(user_prompt):
    m = re.search(r'\[Story\]\n(.*?)\n\n\[Question\]', user_prompt, re.DOTALL)
    return m.group(1) if m else user_prompt[:200]

df_g = df.copy()
df_g['story'] = df_g['user_prompt'].apply(extract_story)
df_g['group_key'] = df_g['task'] + '||' + df_g['story']

groups = df_g['group_key'].values
gss1 = GroupShuffleSplit(n_splits=1, test_size=TEST_RATIO, random_state=42)
trainval_idx, test_idx = next(gss1.split(df_g, groups=groups))
trainval_df_g = df_g.iloc[trainval_idx].reset_index(drop=True)
gss2 = GroupShuffleSplit(n_splits=1, test_size=VAL_RATIO_OF_TRAIN, random_state=42)
train_idx2, val_idx2 = next(gss2.split(trainval_df_g, groups=trainval_df_g['group_key'].values))
train_df_g = trainval_df_g.iloc[train_idx2].reset_index(drop=True)
val_df_g   = trainval_df_g.iloc[val_idx2].reset_index(drop=True)
test_df_g  = df_g.iloc[test_idx].reset_index(drop=True)

print(f'✅ Group split: Train={len(train_df_g)} Val={len(val_df_g)} Test={len(test_df_g)}')
overlap_check = set(zip(train_df_g['task'], train_df_g['story'])) & set(zip(test_df_g['task'], test_df_g['story']))
print(f'✅ train∩test story 겹침: {len(overlap_check)}개 (0이어야 정상)')
assert len(overlap_check) == 0, '⚠️ group split인데 여전히 겹침이 있습니다 — group_key 정의를 확인하세요'


In [ ]:
print('='*60); print('🔥 Group-level split 재학습'); print('='*60)
g_model, g_tokenizer = load_fresh_model()
g_model = FastLanguageModel.get_peft_model(
    g_model, r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16, lora_dropout=0.0, bias='none',
    use_gradient_checkpointing='unsloth', random_state=42,
)
_gpu = torch.cuda.get_device_name(0)
_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
if 'A100' in _gpu or _vram >= 35: TRAIN_BS, GRAD_ACCUM, EVAL_BS = 8, 1, 16
elif 'L4' in _gpu or _vram >= 22: TRAIN_BS, GRAD_ACCUM, EVAL_BS = 4, 2, 8
else: TRAIN_BS, GRAD_ACCUM, EVAL_BS = 2, 4, 4

train_texts_g = [to_sft_text(g_tokenizer, r['user_prompt'], r['answer']) for r in train_df_g.to_dict('records')]
val_texts_g   = [to_sft_text(g_tokenizer, r['user_prompt'], r['answer']) for r in val_df_g.to_dict('records')]
train_ds_g = Dataset.from_dict({'text': train_texts_g})
val_ds_g   = Dataset.from_dict({'text': val_texts_g})

g_trainer = SFTTrainer(
    model=g_model, tokenizer=g_tokenizer, train_dataset=train_ds_g, eval_dataset=val_ds_g,
    dataset_text_field='text', max_seq_length=MAX_SEQ_LENGTH, dataset_num_proc=2, packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=TRAIN_BS, per_device_eval_batch_size=EVAL_BS,
        gradient_accumulation_steps=GRAD_ACCUM, warmup_ratio=0.03, num_train_epochs=NUM_EPOCHS,
        learning_rate=2e-4, fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
        logging_steps=20, eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
        metric_for_best_model='eval_loss', greater_is_better=False, save_total_limit=2,
        optim='adamw_8bit', weight_decay=0.01, lr_scheduler_type='cosine', seed=42,
        output_dir='outputs_groupsplit', report_to='none',
    ),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
)
g_trainer.train()

# base(zero-shot)도 이 group test set 기준으로 다시 평가해야 공정 비교 (원래 base 결과는 item-level test set 기준이었으므로)
base_model_g, base_tok_g = load_fresh_model()
base_res_g = evaluate(test_df_g.to_dict('records'), base_model_g, base_tok_g, desc='group-test-base')
base_acc_g = base_res_g['correct'].mean()*100
del base_model_g, base_tok_g
torch.cuda.empty_cache()

tuned_res_g = evaluate(test_df_g.to_dict('records'), g_model, g_tokenizer, desc='group-test-tuned')
tuned_acc_g = tuned_res_g['correct'].mean()*100

base_res_g.to_csv(f'{TIER3_DIR}/groupsplit_base_ToMBench.csv', index=False)
tuned_res_g.to_csv(f'{TIER3_DIR}/groupsplit_finetuned_ToMBench.csv', index=False)

print(f'\n📊 [Group split — 누수 없는 test set] base={base_acc_g:.2f}%  tuned={tuned_acc_g:.2f}%  Δ={tuned_acc_g-base_acc_g:+.2f}pp')
print(f'📊 [원래 item-level split] base=61.52%  tuned=76.13%  Δ=+14.61pp')
print('\n→ 두 Δ가 비슷하면 누수의 영향이 작다는 뜻, group split Δ가 뚜렷이 작으면 원래 결과 일부가 누수 때문이라는 뜻입니다.')

with open(f'{TIER3_DIR}/groupsplit_summary.json','w') as f:
    json.dump({'base_acc': base_acc_g, 'tuned_acc': tuned_acc_g, 'delta': tuned_acc_g-base_acc_g,
               'n_train': len(train_df_g), 'n_val': len(val_df_g), 'n_test': len(test_df_g),
               'original_item_level_delta': 14.61}, f, indent=2)

del g_model, g_tokenizer, g_trainer
torch.cuda.empty_cache()


## 🎯 [6/7] [중요 F] Few-shot + CoT 베이스라인 (학습 없이, base 모델)

ToMBench train split에서 3개 예시를 뽑아 **간단한 근거(CoT) + 정답**을 few-shot으로 제공합니다. 서론의 "SLM에서 CoT는 오히려 성능을 해친다"는 주장을 직접 실험으로 검증합니다.

In [ ]:
FEWSHOT_IDX = [0, 1, 2]   # train_df에서 고정된 3개 (재현성용, 필요시 바꿔도 됨)

def build_cot_fewshot(tokenizer):
    exemplars = train_df.iloc[FEWSHOT_IDX]
    msgs = []
    for _, r in exemplars.iterrows():
        reasoning = (f"Let's reason step by step: I track who has access to which information in the story, "
                     f"then check what the relevant character would infer given only what they observed. "
                     f"This points to option {r['answer']}.")
        msgs.append({'role':'user','content': r['user_prompt']})
        msgs.append({'role':'assistant','content': f"{reasoning} [[{r['answer']}]]"})
    return msgs

def build_format_only_fewshot(tokenizer):
    # [C3] 포맷통제: 추론 없이 파인튜닝과 동일한 '[[X]]' 만 보여줌
    exemplars = train_df.iloc[FEWSHOT_IDX]
    msgs = []
    for _, r in exemplars.iterrows():
        msgs.append({'role':'user','content': r['user_prompt']})
        msgs.append({'role':'assistant','content': f"[[{r['answer']}]]"})
    return msgs

print('✅ few-shot 빌더 준비 (F: CoT 3-shot, C3: 포맷 전용 3-shot)')
print('\n예시 (F, 1개):')
print(build_cot_fewshot(None)[1]['content'][:200])


In [ ]:
print('='*60); print('🎯 [F] Few-shot + CoT 베이스라인 평가 (base 모델, 학습 없음)'); print('='*60)
base_model, base_tokenizer = load_fresh_model()
fewshot_cot = build_cot_fewshot(base_tokenizer)

fewshot_results = {}
for name, recs in eval_sets.items():
    res_df = evaluate(recs, base_model, base_tokenizer, desc=f'F-fewshot-cot-{name}', few_shot_prefix=fewshot_cot)
    res_df.to_csv(f'{TIER3_DIR}/F_fewshot_cot_{name}.csv', index=False)
    acc = res_df['correct'].mean()*100
    fewshot_results[name] = acc
    print(f'  {name}: zero-shot base={BASE_ACC.get(name, float("nan")):.2f}%  →  few-shot+CoT base={acc:.2f}%  (Δ={acc-BASE_ACC.get(name,0):+.2f}pp)')

with open(f'{TIER3_DIR}/F_fewshot_cot_summary.json','w') as f:
    json.dump(fewshot_results, f, indent=2, ensure_ascii=False)


## 🧩 [7/7] [C3] Base 모델 + 포맷통제 few-shot (추론 없이 출력형식만)

파인튜닝 성능 향상이 '포맷 학습'인지 '진짜 추론 향상'인지 분리하기 위한 대조군입니다. 같은 3개 예시를 추론 없이 정답만 보여줍니다.

In [ ]:
print('='*60); print('🧩 [C3] Base 모델 + 포맷통제 few-shot 평가'); print('='*60)
fewshot_format_only = build_format_only_fewshot(base_tokenizer)

format_only_results = {}
for name, recs in eval_sets.items():
    res_df = evaluate(recs, base_model, base_tokenizer, desc=f'C3-format-only-{name}', few_shot_prefix=fewshot_format_only)
    res_df.to_csv(f'{TIER3_DIR}/C3_format_only_{name}.csv', index=False)
    acc = res_df['correct'].mean()*100
    format_only_results[name] = acc
    print(f'  {name}: zero-shot base={BASE_ACC.get(name, float("nan")):.2f}%  →  format-only-fewshot base={acc:.2f}%  (Δ={acc-BASE_ACC.get(name,0):+.2f}pp)')

with open(f'{TIER3_DIR}/C3_format_only_summary.json','w') as f:
    json.dump(format_only_results, f, indent=2, ensure_ascii=False)

del base_model, base_tokenizer
torch.cuda.empty_cache()


## 📋 최종 비교표 — 논문 F/C3 반영용

In [ ]:
final_rows = []
for ds in eval_sets.keys():
    final_rows.append({
        'Dataset': ds,
        'Zero-shot base': BASE_ACC.get(ds),
        'Few-shot+CoT base (F)': fewshot_results.get(ds),
        'Format-only few-shot base (C3)': format_only_results.get(ds),
        'Fine-tuned seed=42': SEED42_TUNED_ACC.get(ds),
        f'Fine-tuned mean (n={len(EXTRA_SEEDS)+1} seeds)': None,
    })
final_df = pd.DataFrame(final_rows)
# 다중 시드 평균 채우기
for i, row in final_df.iterrows():
    ds = row['Dataset']
    accs = [SEED42_TUNED_ACC[ds]] + [multi_seed_results[s][ds] for s in EXTRA_SEEDS if ds in multi_seed_results.get(s, {})]
    final_df.loc[i, f'Fine-tuned mean (n={len(EXTRA_SEEDS)+1} seeds)'] = np.mean(accs)

print(final_df.to_string(index=False))
final_df.to_csv(f'{TIER3_DIR}/FINAL_tier3_comparison.csv', index=False)
print(f'\n💾 전체 Tier3 결과: {TIER3_DIR}/ 에 저장됨. 이 폴더를 통째로 다운로드해서 Claude에게 전달하면 논문에 반영합니다.')
